# 06 — Tableau Dashboard Prep

Exports **aggregated, summary-level** tables for the Tableau dashboard — not row-level
data. This matters because `HO_infxn_analysis.csv` is PhysioNet credentialed-access data
under a DUA (see [DATA_ACCESS.md](../DATA_ACCESS.md)); row-level extracts must stay local
and out of any file that might get shared or published alongside the dashboard.

Exports go to `tableau/` (git-ignored for any row-level `.hyper`/`.tde` extracts — the
aggregate CSVs below are small enough to be safe to version, but double-check DUA terms
before making the dashboard itself public).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

TABLEAU_DIR = Path("../tableau")
TABLEAU_DIR.mkdir(exist_ok=True)

## Environmental: MRSA acquisition rate by colonization-pressure decile

In [2]:
env = env.copy()
env["mrsa_cp_decile"] = pd.qcut(env["MRSA_cp"], 10, labels=False, duplicates="drop")

cp_by_decile = (
    env.groupby("mrsa_cp_decile")
    .agg(n=("group_binary", "size"), acquisition_rate=("group_binary", "mean"),
         mean_mrsa_cp=("MRSA_cp", "mean"))
    .reset_index()
)
cp_by_decile.to_csv(TABLEAU_DIR / "env_mrsa_cp_by_decile.csv", index=False)
cp_by_decile

,mrsa_cp_decile,n,acquisition_rate,mean_mrsa_cp
0,0,1050,0.317143,0.199624
1,1,349,0.289398,0.885548
2,2,350,0.300000,1.396625
3,3,350,0.300000,1.833267
4,4,349,0.332378,2.445800
5,5,350,0.308571,3.080034
6,6,350,0.337143,3.941581
7,7,350,0.328571,5.822920


## Patient: MRSA acquisition rate by antibiotic class exposure

In [3]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

rows = []
for c in abx_cols:
    exposed = pat[pat[c] > 0]
    unexposed = pat[pat[c] == 0]
    rows.append({
        "antibiotic_class": c.replace("_0_60", ""),
        "n_exposed": len(exposed),
        "acquisition_rate_exposed": exposed["group_binary"].mean(),
        "n_unexposed": len(unexposed),
        "acquisition_rate_unexposed": unexposed["group_binary"].mean(),
    })
abx_summary = pd.DataFrame(rows).sort_values("acquisition_rate_exposed", ascending=False)
abx_summary.to_csv(TABLEAU_DIR / "pat_mrsa_rate_by_abx_class.csv", index=False)
abx_summary

,antibiotic_class,n_exposed,acquisition_rate_exposed,n_unexposed,acquisition_rate_unexposed
8,glycopeptide,11,0.363636,3747,0.293034
11,lincosamide,6,0.333333,3752,0.293177
12,macrolide,35,0.285714,3723,0.293312
6,fluoroquinolone,96,0.270833,3662,0.293829
1,extended_spectrum_penicillin,49,0.244898,3709,0.293880
13,tetracycline,49,0.244898,3709,0.293880
3,cephalosporin,90,0.244444,3668,0.294438
0,penicillin,23,0.173913,3735,0.293976
4,extended_spectrum_cephalosporin,7,0.142857,3751,0.293522
10,anti_anaerobe,7,0.142857,3751,0.293522


## Model results: odds ratios for the dashboard forest plot

In [4]:
import pickle

with open("../reports/logit_environmental.pkl", "rb") as f:
    logit_env = pickle.load(f)
with open("../reports/logit_patient_adjusted.pkl", "rb") as f:
    logit_pat_adj = pickle.load(f)


def or_table(model, source_label):
    params = model.params
    conf = model.conf_int()
    conf.columns = ["ci_low", "ci_high"]
    out = np.exp(pd.concat([params, conf], axis=1).rename(columns={0: "coef"}))
    out.columns = ["OR", "ci_low", "ci_high"]
    out["source"] = source_label
    out["variable"] = out.index
    return out.reset_index(drop=True)


or_all = pd.concat([
    or_table(logit_env, "environmental"),
    or_table(logit_pat_adj, "patient"),
], ignore_index=True)
or_all = or_all[or_all["variable"] != "const"]
or_all.to_csv(TABLEAU_DIR / "model_odds_ratios.csv", index=False)
or_all

,OR,ci_low,ci_high,source,variable
1,0.980872,0.968086,0.993827,environmental,DS_Entero_cp
2,0.975001,0.943125,1.007955,environmental,ESBL_cp
3,0.989352,0.935857,1.045905,environmental,CDiff_cp
4,0.986801,0.952791,1.022026,environmental,VSE_cp
5,1.048148,0.987325,1.112718,environmental,VRE_cp
6,0.994564,0.964725,1.025326,environmental,MSSA_cp
7,1.068786,1.015935,1.124385,environmental,MRSA_cp
8,1.020391,0.971450,1.071797,environmental,DS_PsA_cp
9,1.020443,0.950961,1.095001,environmental,DR_PsA_cp
10,0.958347,0.763985,1.202157,environmental,any_surgery


## Demographic/summary table for dashboard filters

In [5]:
demo_summary = pd.concat([
    env.assign(analysis="environmental"),
    pat.assign(analysis="patient"),
])[["analysis", "group", "age", "sex", "duration"]]

demo_summary.groupby(["analysis", "group"]).agg(
    n=("age", "size"), mean_age=("age", "mean"), mean_duration=("duration", "mean")
).to_csv(TABLEAU_DIR / "cohort_summary.csv")
pd.read_csv(TABLEAU_DIR / "cohort_summary.csv")

,analysis,group,n,mean_age,mean_duration
0,environmental,case,1101,58.798183,7.558583
1,environmental,control,2397,57.642011,4.655611
2,patient,case,1102,62.289474,7.856534
3,patient,control,2656,66.307191,4.888517


## Dashboard sketch

Planned Tableau views (built from the CSVs above, not raw data):

1. **Acquisition rate vs. MRSA colonization-pressure decile** (line/bar) — environmental arm.
2. **Acquisition rate by antibiotic class, exposed vs. unexposed** (grouped bar) — patient arm.
3. **Forest plot of odds ratios** across both models, side by side, to visually answer the
   "which factor matters more" question.
4. **Cohort summary table** (age, sex, duration by arm/group) as dashboard context/filters.